In [1]:
# Instalaciones
%pip install -qU langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters
%pip install -qU transformers accelerate bitsandbytes sentence-transformers pypdf faiss-cpu streamlit

print("\n--- INSTALACIÓN COMPLETADA. ---\n")


--- INSTALACIÓN COMPLETADA. ---



In [2]:
%%writefile app.py
import streamlit as st
import torch
import gc
import hashlib
import tempfile
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Optimización de memoria CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

st.set_page_config(page_title="Copiloto & QA de Traducción", page_icon="🌐", layout="wide")
st.title("🌐 Copiloto y Auditor QA para Traducción Especializada")

if "messages" not in st.session_state:
    st.session_state.messages = []

# 1. Diccionario de especialidades
DOMINIOS_ESPECIALIDAD = {
    "🎮 Localización de Videojuegos y Software": (
        "Eres un Especialista Senior en Localización de Videojuegos (GameLoc) e Ingeniería de Software. "
        "Presta máxima atención a variables de código, etiquetas dinámicas de género, límites de caracteres (UI), "
        "terminología de certificación de consolas (TRC/TCR), tono de facciones y adaptación creativa (transcreación)."
    ),
    "🩺 Traducción Médica y Farmacéutica (Life Sciences)": (
        "Eres un Revisor Senior de Traducción Médica y Dispositivos Biomédicos (normativa ISO 13485 y MDR). "
        "Prioriza el rigor clínico, la nomenclatura anatómica exacta, las advertencias de seguridad, "
        "el formato numérico del Sistema Internacional y la desambiguación estricta de términos clínicos."
    ),
    "⚖️ Traducción Jurídica, Legal y Financiera": (
        "Eres un Traductor Jurado y Experto en Traducción Jurídico-Financiera. "
        "Presta especial atención a la fraseología contractual, fórmulas solemnes, asimetrías entre sistemas jurídicos "
        "(Common Law vs. Derecho Civil), siglas societarias y criterios de no traducción de nombres de instituciones."
    ),
    "⚙️ Traducción Técnica, Industrial y Patentes": (
        "Eres un Ingeniero y Traductor Técnico Senior especializado en manuales industriales, automoción y patentes. "
        "Enfócate en la coherencia terminológica de piezas mecánicas, unidades de medida, "
        "imperativos o infinitivos en instrucciones operativas y alertas de riesgo laboral."
    ),
    "📚 General / Multidominio": (
        "Eres un Gestor de Proyectos de Traducción y Lingüista Senior adaptable a cualquier sector. "
        "Analiza con precisión las reglas ortotipográficas, preferencias de estilo del cliente y glosarios adjuntos."
    )
}

instruccion_dominio = DOMINIOS_ESPECIALIDAD["🎮 Localización de Videojuegos y Software"]

# 2. Carga de modelos en caché
@st.cache_resource
def cargar_modelos_base():

    # Liberar memoria residual
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 1. Embeddings en CPU
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={"device": "cpu"}
    )

    # 2. Cuantización a 4 bits
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    model_id = "Qwen/Qwen2.5-7B-Instruct"

    # 3. Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # 4. Cargar modelo
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    # Comprobar cuantización
    print("Modelo cargado en 4 bits:", model.is_loaded_in_4bit)
    print("Memoria del modelo:", round(
        model.get_memory_footprint() / (1024**3), 2
    ), "GB")

    # 5. Pipeline de generación
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False,
        batch_size=1
    )

    llm = HuggingFacePipeline(pipeline=pipe)

    return embeddings, llm

with st.spinner("Cargando motor Qwen 2.5 (7B-Instruct en 4-bits)..."):
    embeddings, llm = cargar_modelos_base()

def extraer_paginas_pdf(archivo_subido):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
        tmp_file.write(archivo_subido.getvalue())
        ruta_temporal = tmp_file.name
    try:
        loader = PyPDFLoader(ruta_temporal)
        documentos = loader.load()
        return documentos
    finally:
        if os.path.exists(ruta_temporal):
            os.remove(ruta_temporal)

def indexar_guia_estilo(archivo_subido, _embeddings):
    documentos = extraer_paginas_pdf(archivo_subido)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    fragmentos = text_splitter.split_documents(documentos)
    vectorstore = FAISS.from_documents(fragmentos, _embeddings)
    return vectorstore, len(documentos), len(fragmentos)

# 3. Barra lateral
with st.sidebar:
    st.header("🎯 1. Ámbito de Especialidad")
    dominio_seleccionado = st.selectbox(
        "Selecciona el sector del proyecto:",
        options=list(DOMINIOS_ESPECIALIDAD.keys())
    )
    instruccion_dominio = DOMINIOS_ESPECIALIDAD[dominio_seleccionado]

    st.divider()
    st.header("📘 2. Guía de Estilo / Glosario")
    archivo_guia = st.file_uploader("Sube el PDF de instrucciones del cliente", type=["pdf"], key="guia")

    if archivo_guia is not None:
        hash_guia = hashlib.sha256(archivo_guia.getvalue()).hexdigest()
        if "ultimo_pdf" not in st.session_state or st.session_state.ultimo_pdf != hash_guia:
            with st.spinner(f"Indexando guía '{archivo_guia.name}' en FAISS..."):
                vectorstore, num_pags, num_chunks = indexar_guia_estilo(archivo_guia, embeddings)
                st.session_state.vectorstore = vectorstore
                st.session_state.ultimo_pdf = hash_guia
                st.session_state.info_doc = f"✅ **{archivo_guia.name}** ({num_pags} págs. | {num_chunks} fragmentos)"
                st.session_state.messages = [
                    {"role": "assistant", "content": f"He indexado la guía **{archivo_guia.name}** bajo el perfil de **{dominio_seleccionado}**. Puedes hacerme consultas aquí o ir a la pestaña de **Auditoría QA**."}
                ]
        st.success(st.session_state.info_doc)
    else:
        st.info("👆 Sube primero la guía de estilo o glosario en PDF.")

    st.divider()
    if st.button("🗑️ Limpiar memoria y chat"):
        st.session_state.messages = []
        torch.cuda.empty_cache()
        gc.collect()
        st.rerun()

# 4. Pantalla principal
if "vectorstore" not in st.session_state or archivo_guia is None:
    st.warning("👈 Sube primero la **Guía de Estilo / Glosario en PDF** en el panel izquierdo para activar el Copiloto y el Auditor QA.")
else:
    retriever = st.session_state.vectorstore.as_retriever(search_kwargs={"k": 10})

    def formatear_documentos(docs):
        return "\n\n---\n\n".join([f"[Página {doc.metadata.get('page', 0) + 1} de la Guía]:\n{doc.page_content}" for doc in docs])

    tab_chat, tab_qa = st.tabs(["💬 Copiloto de Consulta (Chat)", "🔍 Auditoría QA de Traducción (Original vs. Traducción)"])

    # ==========================================
    # PESTAÑA 1: CHAT DE CONSULTA DE LA GUÍA
    # ==========================================
    with tab_chat:
        plantilla_rag = """<|im_start|>system
{rol_experto}
Tu objetivo es responder a las dudas del traductor basándote ESTRICTAMENTE en el contexto de la guía de estilo proporcionado a continuación.
Cita siempre entre corchetes la página exacta de donde extraes la norma o el término (ej. [Página X]). Si la información no aparece en el contexto, responde exactamente: "No he encontrado esta especificación en el documento proporcionado."

CONTEXTO DE LA GUÍA DE ESTILO:
{context}
<|im_end|>
<|im_start|>user
Consulta del traductor: {question}
<|im_end|>
<|im_start|>assistant
"""
        prompt_chat = PromptTemplate.from_template(plantilla_rag).partial(rol_experto=instruccion_dominio)

        cadena_rag = (
            {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
            | prompt_chat
            | llm
            | StrOutputParser()
        )

        # 1. Creamos un contenedor con altura fija (ej. 250 píxeles) para que haga scroll
        contenedor_mensajes = st.container(height=250)

        # 2. Pintamos el historial DENTRO del contenedor
        with contenedor_mensajes:
            for msg in st.session_state.messages:
                with st.chat_message(msg["role"]):
                    st.markdown(msg["content"])

        # 3. El chat_input queda justo debajo del contenedor (fijo a la vista)
        if prompt_usuario := st.chat_input("Escribe aquí tu duda sobre la guía de estilo o glosario..."):
            st.session_state.messages.append({"role": "user", "content": prompt_usuario})

            # 4. Mostramos los nuevos mensajes también DENTRO del contenedor
            with contenedor_mensajes:
                with st.chat_message("user"):
                    st.markdown(prompt_usuario)

                with st.chat_message("assistant"):
                    with st.spinner("Analizando guía de estilo..."):
                        torch.cuda.empty_cache()
                        gc.collect()
                        respuesta = cadena_rag.invoke(prompt_usuario).strip()
                        docs_recuperados = retriever.invoke(prompt_usuario)
                        st.markdown(respuesta)

                        with st.expander("📄 Ver fragmentos originales recuperados de la Guía (Evidencia RAG)"):
                            for i, doc in enumerate(docs_recuperados):
                                pag = doc.metadata.get('page', 0) + 1
                                st.markdown(f"**Fragmento {i+1} (Página {pag}):**")
                                st.caption(doc.page_content)
                                st.divider()

            st.session_state.messages.append({"role": "assistant", "content": respuesta})

    # ==========================================
    # PESTAÑA 2: AUDITORÍA QA (SOURCE VS TARGET)
    # ==========================================
    with tab_qa:
        st.subheader("🔍 Verificación Automática de Cumplimiento Normativo (LQA)")
        st.markdown(
            "Sube el **PDF Original (Source)** y el **PDF Traducido (Target)**, "
            "o pega un fragmento manualmente. El sistema cruzará ambos textos "
            "con la Guía de Estilo para detectar incumplimientos."
        )

        modo_qa = st.radio(
            "Selecciona el modo de revisión:",
            [
                "📄 Subir PDF Original y PDF Traducido",
                "✍️ Pegar texto Original y Traducido manualmente"
            ],
            horizontal=True
        )

        plantilla_auditoria = """<|im_start|>system
{rol_experto}

Actúas como un AUDITOR SENIOR DE CALIDAD DE TRADUCCIÓN
(Linguistic Quality Assurance - LQA), especializado en
localización de videojuegos y cumplimiento de guías de estilo.

Tu objetivo es realizar una auditoría exhaustiva del TEXTO
TRADUCIDO (TARGET), contrastándolo con el TEXTO ORIGINAL
(SOURCE) y con las reglas aplicables de la GUÍA DE ESTILO
que se te proporcionen.

Tu prioridad es detectar cualquier incumplimiento demostrable.
No apruebes una traducción únicamente porque sea gramatical,
fluida o transmita una idea similar al original.

==================================================
FASE 1. ANÁLISIS DE LAS REGLAS DE LA GUÍA
==================================================

Antes de evaluar la traducción:

1. Identifica las reglas aplicables al texto que vas a auditar.

2. Presta especial atención a:
   - Terminología obligatoria y traducciones prohibidas.
   - Glosarios y equivalencias terminológicas.
   - Tratamiento formal e informal (tú, usted, vosotros, etc.).
   - Convenciones específicas de localización de videojuegos.
   - Nombres de botones y controles de PlayStation, Xbox y Nintendo.
   - Convenciones de puntuación, mayúsculas y ortografía.
   - Formatos numéricos, fechas, unidades y símbolos.
   - Calcos léxicos y construcciones sintácticas prohibidas.
   - Restricciones de longitud, variables, etiquetas y elementos
     que no deben traducirse.

3. Distingue entre reglas obligatorias, recomendaciones y ejemplos.
   No conviertas una recomendación en una obligación.

4. Ten en cuenta el contexto, la plataforma y el tipo de contenido.

5. Utiliza únicamente las reglas recuperadas de la guía para
   justificar incumplimientos específicos. No inventes normas.

IMPORTANTE: las reglas proporcionadas pueden ser una selección
recuperada mediante búsqueda. No presupongas que has recibido
la totalidad de la guía.

==================================================
FASE 2. COMPARACIÓN SOURCE-TARGET
==================================================

Compara ambos textos de forma sistemática, segmento por segmento
y unidad de significado por unidad de significado.

Comprueba obligatoriamente:

A. FIDELIDAD Y COMPLETITUD
   - Omisiones de palabras, frases, instrucciones o información.
   - Adiciones injustificadas.
   - Cambios de significado, intención o matiz.
   - Errores de interpretación.
   - Inconsistencias entre el original y la traducción.

B. TERMINOLOGÍA
   - Comprueba cada término relevante frente al glosario.
   - Detecta términos que deberían haberse traducido de una
     forma específica.
   - Detecta variantes terminológicas no autorizadas.
   - Comprueba la coherencia terminológica en todo el texto.

C. REGISTRO Y TRATAMIENTO
   - Comprueba el uso correcto de tú, usted, vosotros, etc.
   - Detecta cambios injustificados de registro.
   - Verifica que se respete el tono establecido en la guía.

D. LOCALIZACIÓN DE VIDEOJUEGOS
   - Comprueba los nombres de botones según la plataforma.
   - Diferencia entre PlayStation, Xbox y Nintendo.
   - Detecta nombres de controles incorrectos o mezclados.
   - Comprueba los términos de interfaz y los nombres oficiales
     de elementos del juego.

E. CALIDAD LINGÜÍSTICA
   - Detecta errores gramaticales y sintácticos.
   - Detecta calcos léxicos y sintácticos del inglés.
   - Identifica falsos amigos y traducciones excesivamente literales.
   - Comprueba la naturalidad del español sin alterar el significado.

F. FORMATO Y CONVENCIONES
   - Comprueba números, fechas, unidades y símbolos.
   - Revisa puntuación, espacios y mayúsculas.
   - Detecta errores en variables, etiquetas y códigos.
   - Comprueba cualquier otra convención específica de la guía.

No confundas una diferencia estilística con un error.
Una traducción puede utilizar una construcción distinta del
original y ser perfectamente válida.

Sin embargo, si existe una equivalencia obligatoria establecida
en la guía, debes exigir su cumplimiento.

==================================================
FASE 3. VERIFICACIÓN DE LOS INCUMPLIMIENTOS
==================================================

Para cada posible error detectado:

1. Cita literalmente el fragmento del TARGET.
2. Indica el fragmento correspondiente del SOURCE, cuando
   sea relevante.
3. Identifica la regla concreta de la guía que se incumple.
4. Cita el número de página exacto, si está disponible.
5. Explica por qué existe un incumplimiento.
6. Proporciona la corrección exacta o una propuesta que cumpla
   la guía de estilo.

No inventes números de página, citas ni reglas.
Si no puedes verificar una referencia, indícalo expresamente.

No marques como error algo que no puedas justificar mediante
el original, la guía o una regla lingüística objetiva.

Si detectas un posible problema, pero no dispones de información
suficiente para confirmarlo, clasifícalo como duda pendiente
de verificación y no como incumplimiento confirmado.

==================================================
FASE 4. CONTROL DE EXHAUSTIVIDAD
==================================================

Antes de emitir el informe:

- Comprueba que has revisado todos los segmentos del TARGET.
- Contrasta las reglas recuperadas que sean aplicables.
- Revisa especialmente omisiones, términos incorrectos,
  botones, formalidad, números y calcos del inglés.
- No detengas la auditoría después de encontrar los primeros errores.
- Evita repetir un mismo incumplimiento varias veces, salvo que
  aparezca en contextos distintos.

Si el texto es demasiado extenso para revisarlo completamente,
indícalo y especifica qué partes no has podido verificar.

No declares una auditoría completa si no has podido revisar
todo el contenido proporcionado.

==================================================
FORMATO DEL INFORME FINAL
==================================================

### 1. ESTADO DE CUMPLIMIENTO

Selecciona uno de estos estados:

- ✅ CORRECTO: no se han detectado incumplimientos confirmados.
- ⚠️ CON ADVERTENCIAS: existen errores leves o dudas que
  requieren revisión.
- ❌ INCUMPLIMIENTO GRAVE: existen errores críticos o graves
  que impiden aprobar la traducción.

### 2. ERRORES DETECTADOS

Para cada error utiliza esta estructura:

**Error [número] — [Categoría]**
- Severidad: Crítica / Grave / Leve.
- SOURCE: "[fragmento original]"
- TARGET: "[fragmento traducido]"
- Regla incumplida: "[cita o descripción precisa]"
- Referencia: [Página X, si está disponible]
- Justificación: explicación concreta del incumplimiento.
- Corrección propuesta: "[propuesta corregida]"

Clasifica los errores según su gravedad:

- CRÍTICA: alteración importante del significado u omisión
  de información esencial.
- GRAVE: incumplimiento terminológico, de botones, de registro
  o de una regla obligatoria de la guía.
- LEVE: errores menores de puntuación, ortografía, formato
  o estilo que no alteran significativamente el significado.

### 3. ASPECTOS CORRECTOS Y VERIFICADOS

Indica brevemente qué aspectos importantes cumplen las
especificaciones de la guía.

No afirmes haber comprobado una categoría si no has podido
verificarla.

### 4. DUDAS PENDIENTES DE VERIFICACIÓN

Incluye aquí posibles problemas que no puedan confirmarse
por falta de contexto, información o reglas recuperadas.

Si no hay ninguno, indícalo expresamente.

### 5. RESULTADO DE LA AUDITORÍA

Resume:
- Número total de errores confirmados.
- Número de errores críticos, graves y leves.
- Número de dudas pendientes.
- Reglas de la guía incumplidas.
- Si el texto está listo para su aprobación o necesita
  una nueva revisión.

REGLA FINAL:

Sé riguroso, objetivo y exhaustivo. Tu función no es validar
el trabajo del traductor, sino detectar cualquier incumplimiento
demostrable y proporcionar una solución concreta.
<|im_end|>
<|im_start|>user
REGLAS Y GLOSARIO RECUPERADOS DE LA GUÍA DE ESTILO:

{reglas_guia}

==================================================
TEXTO ORIGINAL (SOURCE):
==================================================

{texto_origen}

==================================================
TEXTO TRADUCIDO (TARGET):
==================================================

{texto_meta}

==================================================

Realiza el informe de auditoría QA siguiendo todas las fases
e instrucciones anteriores.
<|im_end|>
<|im_start|>assistant
"""

        prompt_qa = PromptTemplate.from_template(plantilla_auditoria).partial(
            rol_experto=instruccion_dominio
        )
        cadena_qa = prompt_qa | llm | StrOutputParser()

        if modo_qa == "📄 Subir PDF Original y PDF Traducido":
            col1, col2 = st.columns(2)
            with col1:
                pdf_origen = st.file_uploader(
                    "1️⃣ Sube el PDF Original (Source)",
                    type=["pdf"],
                    key="src_pdf"
                )
            with col2:
                pdf_meta = st.file_uploader(
                    "2️⃣ Sube el PDF Traducido (Target)",
                    type=["pdf"],
                    key="tgt_pdf"
                )

            if pdf_origen is not None and pdf_meta is not None:
                pags_origen = extraer_paginas_pdf(pdf_origen)
                pags_meta = extraer_paginas_pdf(pdf_meta)

                if len(pags_origen) != len(pags_meta):
                    st.warning(
                        f"⚠️ Los documentos tienen diferente número de páginas: "
                        f"{len(pags_origen)} originales y {len(pags_meta)} traducidas. "
                        "Se auditarán únicamente las páginas coincidentes por posición. "
                        "Comprueba que ambas versiones estén alineadas."
                    )

                total_pags = min(len(pags_origen), len(pags_meta))

                if total_pags == 0:
                    st.error("No se han podido extraer páginas de uno o ambos PDF.")
                else:
                    st.info(
                        f"📊 Documentos cargados: **{len(pags_origen)}** págs. en original | "
                        f"**{len(pags_meta)}** págs. en traducción."
                    )

                    if total_pags > 1:
                        max_pags_auditar = st.slider(
                            "Número de páginas a auditar en este lote:",
                            min_value=1,
                            max_value=total_pags,
                            value=min(3, total_pags)
                        )
                    else:
                        max_pags_auditar = 1
                        st.info("📄 Hay una única página coincidente. Se auditará automáticamente.")

                    if st.button("🚀 Iniciar Auditoría QA por Lotes", type="primary"):
                        barra_progreso = st.progress(0)

                        for i in range(max_pags_auditar):
                            txt_src = pags_origen[i].page_content.strip()
                            txt_tgt = pags_meta[i].page_content.strip()

                            with st.expander(f"📋 Informe QA - Página {i + 1}", expanded=True):
                                with st.spinner(
                                    f"Auditando página {i + 1} de {max_pags_auditar}..."
                                ):
                                    if torch.cuda.is_available():
                                        torch.cuda.empty_cache()
                                    gc.collect()

                                    consulta_busqueda = f"""
Categorías de reglas que deben comprobarse:
- Fidelidad y completitud de la traducción.
- Terminología y glosario.
- Tratamiento formal e informal.
- Botones y controles de videojuegos.
- Calcos léxicos y sintácticos.
- Gramática y ortografía.
- Formato numérico, puntuación y convenciones.

TEXTO ORIGINAL:
{txt_src}

TEXTO TRADUCIDO:
{txt_tgt}
"""
                                    docs_reglas = retriever.invoke(consulta_busqueda)
                                    reglas_formateadas = formatear_documentos(docs_reglas)

                                    informe = cadena_qa.invoke({
                                        "reglas_guia": reglas_formateadas,
                                        "texto_origen": txt_src,
                                        "texto_meta": txt_tgt
                                    }).strip()

                                    st.markdown(informe)

                                    with st.popover(
                                        "🔍 Ver textos comparados y reglas aplicadas"
                                    ):
                                        st.markdown("**Texto Original (extracto):**")
                                        st.caption(txt_src[:600])
                                        st.markdown("**Texto Traducido (extracto):**")
                                        st.caption(txt_tgt[:600])
                                        st.markdown("**Normas de la Guía consultadas por el RAG:**")
                                        st.caption(reglas_formateadas)

                            barra_progreso.progress((i + 1) / max_pags_auditar)

                        st.success("✅ ¡Auditoría QA completada!")

        else:
            col_t1, col_t2 = st.columns(2)
            with col_t1:
                txt_manual_src = st.text_area(
                    "Pega el fragmento Original (Inglés):",
                    height=180,
                    placeholder="Ej: Press the Cross button to cast a Ward spell on your party..."
                )
            with col_t2:
                txt_manual_tgt = st.text_area(
                    "Pega la propuesta de Traducción (Español):",
                    height=180,
                    placeholder="Ej: Pulsa el botón X para tirar un hechizo de Escudo sobre tu grupo..."
                )

            if st.button("🔍 Auditar Fragmento", type="primary"):
                if txt_manual_src.strip() and txt_manual_tgt.strip():
                    with st.spinner(
                        "Cruzando traducción con la guía de estilo y el glosario..."
                    ):
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        gc.collect()

                        consulta_busqueda = f"""
Categorías de reglas que deben comprobarse:
- Fidelidad y completitud.
- Terminología y glosario.
- Tratamiento formal e informal.
- Botones y controles de videojuegos.
- Calcos léxicos y sintácticos.
- Gramática y ortografía.
- Formato y convenciones.

TEXTO ORIGINAL:
{txt_manual_src}

TEXTO TRADUCIDO:
{txt_manual_tgt}
"""
                        docs_reglas = retriever.invoke(consulta_busqueda)
                        reglas_formateadas = formatear_documentos(docs_reglas)

                        informe = cadena_qa.invoke({
                            "reglas_guia": reglas_formateadas,
                            "texto_origen": txt_manual_src,
                            "texto_meta": txt_manual_tgt
                        }).strip()

                        st.markdown("### 📋 Resultado de la Auditoría QA")
                        st.markdown(informe)

                        with st.expander(
                            "📄 Ver reglas de la guía que se han utilizado para evaluar"
                        ):
                            st.caption(reglas_formateadas)
                else:
                    st.warning(
                        "Por favor, rellena tanto el texto original como el texto traducido."
                    )


Overwriting app.py


In [3]:
import subprocess
import time
import re

# 1. Cerramos cualquier proceso previo por seguridad
!pkill -9 streamlit
!pkill -9 cloudflared
time.sleep(2)

# 2. Descargamos Cloudflare Tunnel
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 3. Arrancamos Streamlit en segundo plano
print("🚀 Iniciando servidor de Streamlit con la GPU 100% libre...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

time.sleep(3)

# 4. Generamos el enlace público
print("🌐 Generando enlace público...")
proceso_tunel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for linea in proceso_tunel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", linea)
    if match:
        url_publica = match.group(0)
        print("\n" + "="*65)
        print(f"🎉 ¡APLICACIÓN LISTA! Entra en este enlace:")
        print(f"👉 {url_publica}")
        print("="*65 + "\n")
        break

🚀 Iniciando servidor de Streamlit con la GPU 100% libre...
🌐 Generando enlace público...

🎉 ¡APLICACIÓN LISTA! Entra en este enlace:
👉 https://hrs-pull-und-shapes.trycloudflare.com

